# Adult Census Income: preprocessing и train/test split

В этом notebook подготавливаем данные к обучению моделей:

- загружаем очищенные данные;
- добавляем согласованные engineered features;
- разделяем данные на `X` и `y`;
- делаем `train_test_split` со `stratify=y`;
- создаём preprocessing pipeline через `ColumnTransformer`.

Обучение моделей будет в следующем notebook.

## Импорты и настройки

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

import sys
from pathlib import Path

sys.path.append(str(Path.cwd()))
sys.path.append(str(Path.cwd() / "src"))

from adult_income_utils import (
    RANDOM_STATE,
    add_features,
    get_feature_lists,
    load_clean_adult_data,
    make_X_y,
    make_preprocessor,
)

pd.set_option("display.max_columns", 100)

## Загрузка данных и feature engineering

Общая логика загрузки, очистки и feature engineering вынесена в `adult_income_utils.py`.

In [ ]:
df_clean = load_clean_adult_data()
df_features = add_features(df_clean)

print("Rows after cleaning:", len(df_clean))
print("Shape after feature engineering:", df_features.shape)

df_features.head()

## Разделение на `X` и `y`

`X` содержит признаки, по которым модель будет делать предсказание.

`y` содержит целевую переменную:

- `0` — доход `<=50K`;
- `1` — доход `>50K`.

In [ ]:
X, y = make_X_y(df_features)

print("X shape:", X.shape)
print("y shape:", y.shape)

y.value_counts(normalize=True).rename("class_share")

## Train/test split

Используем `stratify=y`, чтобы сохранить примерно одинаковую долю классов `<=50K` и `>50K` в train и test.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

In [ ]:
class_balance = pd.DataFrame({
    "full": y.value_counts(normalize=True).sort_index(),
    "train": y_train.value_counts(normalize=True).sort_index(),
    "test": y_test.value_counts(normalize=True).sort_index(),
})

class_balance

Доля классов в train и test почти совпадает с полной выборкой. Это важно для несбалансированной бинарной классификации.

## Списки числовых и категориальных признаков

In [ ]:
numerical_features, categorical_features = get_feature_lists(X_train)

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

## Preprocessing pipeline

Для числовых признаков:

- заполняем пропуски медианой;
- масштабируем через `StandardScaler`.

Для категориальных признаков:

- заполняем пропуски категорией `Unknown`;
- кодируем категории через `OneHotEncoder`.

В моделировании этот `preprocessor` нужно помещать внутрь общего `Pipeline` вместе с моделью.

In [ ]:
preprocessor = make_preprocessor(numerical_features, categorical_features)

preprocessor

## Проверка preprocessing

Проверим, что `preprocessor` обучается только на train и может преобразовать train/test в числовые матрицы.

In [ ]:
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

print("Preprocessed train shape:", X_train_preprocessed.shape)
print("Preprocessed test shape:", X_test_preprocessed.shape)

In [ ]:
feature_names = preprocessor.get_feature_names_out()

print("Number of features after preprocessing:", len(feature_names))
feature_names[:20]

## Решение

Для дальнейшего моделирования используем:

- `X_train`, `X_test`, `y_train`, `y_test`;
- `numerical_features` и `categorical_features`;
- `preprocessor` на основе `ColumnTransformer`.

Test set не используется для выбора модели или гиперпараметров. Он нужен только для финальной оценки.